In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_entity_relationships_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "EntityRelationshipId    string,"
        #     "SourceSystem  string,"
        #     "PortfolioKey  string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp   timestamp,"
        #     "BatchId  string,"
        #     "EntityRelationshipKey   string,"
        #     "EntityKey    string,"
        #     "RelatedEntityKey string,"
        #     "EntityRelationshipType    string,"
        #     "EntityRelationshipSubType    string,"
        #     "EntityType    string,"
        #     "RelatedEntityType    string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.EntityRelationships"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== DIM_CASES_STATIC ==================
        DIM_CASES_STATIC_extra_filters = None

        if DIM_CASES_STATIC_extra_filters is not None:
            DIM_CASES_STATIC_extra_filters.append(date_filter)
            DIM_CASES_STATIC_final_filters = " AND ".join(
                DIM_CASES_STATIC_extra_filters
            )
        else:
            DIM_CASES_STATIC_final_filters = date_filter

        DIM_CASES_STATIC_selected_fields = [
            "CASE_ACCOUNT_FOR_SEARCH",
            "CASE_CODE",
            "DEBT_CODE",
            "businessdate",
        ]

        DIM_CASES_STATIC = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_CASES_STATIC",
            DIM_CASES_STATIC_final_filters,
            DIM_CASES_STATIC_selected_fields,
        )

        # ================== FS_CASES_SNAP ==================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = [
            "CAEX_CUSTOM_TEXT_30",
            "CASE_CODE",
            "SPET_CODE",
        ]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        # ================== FS_CASES_SNAP_EXTRA ==================

        FS_CASES_SNAP_EXTRA_extra_filters = None

        if FS_CASES_SNAP_EXTRA_extra_filters is not None:
            FS_CASES_SNAP_EXTRA_extra_filters.append(date_filter)
            FS_CASES_SNAP_EXTRA_final_filters = " AND ".join(FS_CASES_SNAP_EXTRA_extra_filters)
        else:
            FS_CASES_SNAP_EXTRA_final_filters = date_filter

        FS_CASES_SNAP_EXTRA_selected_fields = [
            "CASE_CODE",
        ]

        FS_CASES_SNAP_EXTRA = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP_EXTRA",
            FS_CASES_SNAP_EXTRA_final_filters,
            FS_CASES_SNAP_EXTRA_selected_fields,
        )

        # ================== FSAD_DEBTS ==================

        FSAD_DEBTS_extra_filters = None

        if FSAD_DEBTS_extra_filters is not None:
            FSAD_DEBTS_extra_filters.append(date_filter)
            FSAD_DEBTS_final_filters = " AND ".join(FSAD_DEBTS_extra_filters)
        else:
            FSAD_DEBTS_final_filters = date_filter

        FSAD_DEBTS_selected_fields = [
            "DEBT_CODE",
        ]

        FSAD_DEBTS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FSAD_DEBTS",
            FSAD_DEBTS_final_filters,
            FSAD_DEBTS_selected_fields,
        )

        # ================== Join Conditions ==================
        DIM_CASES_STATIC_FS_CASES_SNAP_join_condition = (
            (DIM_CASES_STATIC["CASE_CODE"] == FS_CASES_SNAP["CASE_CODE"])
            & (FS_CASES_SNAP["CAEX_CUSTOM_TEXT_30"].isNotNull())
        )

        FS_CASES_SNAP_portfolio_join_condition = (
            FS_CASES_SNAP["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        DIM_CASES_STATIC_FS_CASES_SNAP_EXTRA_join_condition = (
            DIM_CASES_STATIC["CASE_CODE"] == FS_CASES_SNAP_EXTRA["CASE_CODE"]
        )

        DIM_CASES_STATIC_FSAD_DEBTS_join_condition = (
            DIM_CASES_STATIC["DEBT_CODE"] == FSAD_DEBTS["DEBT_CODE"]
        )

        # ================== Joins ==================

        result_df = (
            DIM_CASES_STATIC.join(
                FS_CASES_SNAP,
                DIM_CASES_STATIC_FS_CASES_SNAP_join_condition,
                "INNER",
            ).join(
                portfolio,
                FS_CASES_SNAP_portfolio_join_condition,
                "left",
            )
            .join(
                FS_CASES_SNAP_EXTRA,
                DIM_CASES_STATIC_FS_CASES_SNAP_EXTRA_join_condition,
                "INNER",
            )
            .join(
                FSAD_DEBTS,
                DIM_CASES_STATIC_FSAD_DEBTS_join_condition,
                "INNER",
            )
        )

        # HashKeys
        result_df_primary_key_fields = [
            FS_CASES_SNAP.CAEX_CUSTOM_TEXT_30,
            DIM_CASES_STATIC.CASE_ACCOUNT_FOR_SEARCH,
            lit("Master - Slave Rltnp"),
            lit("qca"),
        ]

        result_df = result_df.withColumn(
            "EntityRelationshipId",
            sha2(concat_ws("", *result_df_primary_key_fields), 256),
        ).withColumn(
            "EntityRelationshipKey",
            concat_ws(
                ";",
                FS_CASES_SNAP.CAEX_CUSTOM_TEXT_30,
                DIM_CASES_STATIC.CASE_ACCOUNT_FOR_SEARCH,
                lit("Master - Slave Rltnp"),
            ),
        )

        # ================== Selection of Final Columns ==================
        result_df_qca = result_df.select(
            result_df.EntityRelationshipId.alias("EntityRelationshipId"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df.EntityRelationshipKey.alias("EntityRelationshipKey"),
            trim(FS_CASES_SNAP.CAEX_CUSTOM_TEXT_30).alias("EntityKey"),
            trim(DIM_CASES_STATIC.CASE_ACCOUNT_FOR_SEARCH).alias("RelatedEntityKey"),
            lit("Master - Slave Rltnp").alias("EntityRelationshipType"),
            lit(None).alias("EntityRelationshipSubType"),
            lit("Exposure").alias("EntityType"),
            lit("Exposure").alias("RelatedEntityType"),
            DIM_CASES_STATIC.businessdate.alias("BusinessDate")
        )

        # ================== Append ==================
        result_df_qca = (
            result_df_qca.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_entity_relationships_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)